In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler

In [2]:
df = pd.read_csv(r"E:\SGP Fake Review\Dataset\reviewer_dataset.csv")

In [3]:
df.columns

Index(['reviewerID', 'avg_rating', 'total_reviews', 'rating_variance',
       'rating_entropy', 'first_review', 'last_review', 'active_days',
       'reviews/day', 'review_burst_7d', 'burst_ratio_7d', '1_star_reviews',
       '%_1Star', '2_star_reviews', '%_2Star', '3_star_reviews', '%_3Star',
       '4_star_reviews', '%_4Star', '5_star_reviews', '%_5Star',
       'unique_products_reviewed', 'total_helpful_votes',
       '%_helpful_per_review', 'avg_days_between_reviews'],
      dtype='object')

In [4]:
df_model = df[df['total_reviews']>=3].copy()

In [5]:
df_model['burst_intensity'] = df_model['review_burst_7d'] * df_model['burst_ratio_7d']

In [6]:
features = [
    "rating_variance",
    "rating_entropy",
    "reviews/day",
    "review_burst_7d",
    "burst_ratio_7d",
    "avg_days_between_reviews",
    "burst_intensity"
]

X_new = df_model[features].copy()

In [7]:
X_new.columns = ['rating_variance', 'rating_entropy', 'reviews_per_day', 'review_burst_7d',
       'burst_ratio_7d', 'avg_days_between_reviews', 'burst_intensity']

In [8]:
X_new.reset_index(drop = True, inplace = True)

In [9]:
X_new.loc[:,"avg_days_between_reviews"] = np.log1p(X_new["avg_days_between_reviews"])


In [10]:
X_new

,rating_variance,rating_entropy,reviews_per_day,review_burst_7d,burst_ratio_7d,avg_days_between_reviews,burst_intensity
0,1.200000,1.370951,0.014164,3,0.600000,4.488636,1.800000
1,0.000000,0.000000,0.294118,3,0.600000,1.609438,1.800000
2,0.000000,0.000000,0.035354,2,0.285714,3.521447,0.571429
3,0.200000,0.721928,0.007874,1,0.200000,5.072044,0.200000
4,1.604040,0.979971,0.022310,3,0.066667,3.846272,0.200000
...,...,...,...,...,...,...,...
9430,2.785714,1.905639,0.002974,1,0.125000,5.952130,0.125000
9431,0.000000,0.000000,0.004525,1,0.333333,5.803628,0.333333
9432,4.000000,1.392147,0.009626,3,0.333333,4.766438,1.000000
9433,0.285714,0.371232,0.005863,4,0.285714,5.216607,1.142857


In [11]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_new)

In [12]:
iso = IsolationForest(
    n_estimators=200,
    contamination=0.02,
    random_state=42
)

_ = iso.fit(X_scaled)
print("Isolation Forest trained successfully.")

Isolation Forest trained successfully.


In [13]:
pred_iso = iso.predict(X_scaled)
scores_iso = iso.decision_function(X_scaled)

score_scaler = MinMaxScaler((0,100))

behavior_score_iso = score_scaler.fit_transform(
    scores_iso.reshape(-1,1)
)

In [14]:
X_new.loc[:,"prediction"] = pred_iso

X_new.loc[:,"decision_score"] = scores_iso

X_new.loc[:,"behavior_score"] = behavior_score_iso

In [15]:
X_new

,rating_variance,rating_entropy,reviews_per_day,review_burst_7d,burst_ratio_7d,avg_days_between_reviews,burst_intensity,prediction,decision_score,behavior_score
0,1.200000,1.370951,0.014164,3,0.600000,4.488636,1.800000,1,0.198859,81.724275
1,0.000000,0.000000,0.294118,3,0.600000,1.609438,1.800000,1,0.109184,59.754776
2,0.000000,0.000000,0.035354,2,0.285714,3.521447,0.571429,1,0.198430,81.619172
3,0.200000,0.721928,0.007874,1,0.200000,5.072044,0.200000,1,0.258341,96.296625
4,1.604040,0.979971,0.022310,3,0.066667,3.846272,0.200000,1,0.180825,77.306020
...,...,...,...,...,...,...,...,...,...,...
9430,2.785714,1.905639,0.002974,1,0.125000,5.952130,0.125000,1,0.158961,71.949709
9431,0.000000,0.000000,0.004525,1,0.333333,5.803628,0.333333,1,0.229467,89.222860
9432,4.000000,1.392147,0.009626,3,0.333333,4.766438,1.000000,1,0.169804,74.606007
9433,0.285714,0.371232,0.005863,4,0.285714,5.216607,1.142857,1,0.214079,85.452854


In [16]:
import joblib

joblib.dump(scaler, "scaler.pkl")              
joblib.dump(iso, "isolation_forest.pkl")       
joblib.dump(score_scaler, "score_scaler.pkl")  

print("Saved: scaler.pkl, isolation_forest.pkl, score_scaler.pkl")

Saved: scaler.pkl, isolation_forest.pkl, score_scaler.pkl


In [17]:
def score_reviewer(reviewer_features: dict, scaler, iso, score_scaler) -> dict:
    """
    reviewer_features must contain the 7 RAW (untransformed) fields:
        rating_variance, rating_entropy, reviews_per_day,
        review_burst_7d, burst_ratio_7d, avg_days_between_reviews, burst_intensity

    Do NOT pre-apply log1p to avg_days_between_reviews — this function does it.
    Pass in the scaler/iso/score_scaler objects explicitly so this can be
    tested against either the in-memory model or a reloaded one.
    """
    row = pd.DataFrame([reviewer_features])
    row = row[["rating_variance", "rating_entropy", "reviews_per_day",
               "review_burst_7d", "burst_ratio_7d",
               "avg_days_between_reviews", "burst_intensity"]]

    row["avg_days_between_reviews"] = np.log1p(row["avg_days_between_reviews"])
    row_scaled = scaler.transform(row)

    pred = iso.predict(row_scaled)[0]
    decision = iso.decision_function(row_scaled)[0]
    behavior = score_scaler.transform([[decision]])[0][0]

    return {
        "prediction": int(pred),
        "decision_score": float(decision),
        "behavior_score": float(behavior),
    }

In [18]:
scaler_check = joblib.load("scaler.pkl")
iso_check = joblib.load("isolation_forest.pkl")
score_scaler_check = joblib.load("score_scaler.pkl")

sample = X_new.iloc[0][["rating_variance", "rating_entropy", "reviews_per_day",
                          "review_burst_7d", "burst_ratio_7d",
                          "avg_days_between_reviews", "burst_intensity"]].copy()
sample["avg_days_between_reviews"] = np.expm1(sample["avg_days_between_reviews"])

result = score_reviewer(sample.to_dict(), scaler_check, iso_check, score_scaler_check)
print(result)
print("Should match X_new.iloc[0]['behavior_score']:", X_new.iloc[0]["behavior_score"])

{'prediction': 1, 'decision_score': 0.19885938202431275, 'behavior_score': 81.72427467074148}
Should match X_new.iloc[0]['behavior_score']: 81.72427467074148


In [21]:
df_model["prediction"] = pred_iso
df_model["decision_score"] = scores_iso
df_model["behavior_score"] = behavior_score_iso

df_model[["reviewerID", "prediction", "decision_score", "behavior_score"]].to_csv(
    "Behaviour Score.csv", index=False
)